# Day 02: Trying to Upload PDFs


## 1. Preparing the PDF Processing Pipeline

### Objective
Prepare the existing FastAPI backend to accept PDF uploads and extract their text.

### Code change
Added the following imports to `backend/Main.py`:

```python
from fastapi import FastAPI, UploadFile, File, HTTPException
from pypdf import PdfReader
```

### Why these imports?

- `UploadFile`: Allows FastAPI to receive uploaded files.
- `File`: Declares a request parameter as a file upload.
- `HTTPException`: Allows the API to return appropriate HTTP errors.
- `PdfReader`: Provides tools to read PDF files and extract their text.

### Design decision
We are extending the existing FastAPI application rather than creating a separate application. The existing `/` and `/health` endpoints should remain unchanged.

### Status
Imports added. The PDF upload endpoint has not yet been implemented or tested.

## 2. Implementing the PDF Upload Endpoint

### Objective
Create an API endpoint that accepts a PDF and extracts its text.

### Code change
Added `POST /upload-pdf` to `backend/Main.py`.

### How it works
1. FastAPI receives the uploaded file.
2. The endpoint checks the declared content type and basic PDF signature.
3. The file contents are passed to `PdfReader`.
4. Text is extracted from each page.
5. The API returns the filename, page count, and extracted text as JSON.

### Concepts learned

- **HTTP POST:** Commonly used to send data to a server.
- **Request validation:** Checking incoming data before processing it.
- **Bytes:** File contents are handled as binary data rather than ordinary text.
- **PDF text extraction:** `pypdf` can extract embedded text, but scanned pages may require OCR.
- **JSON response:** Structured data returned by the API.

### Status
Endpoint implemented. Testing is pending.

### Limitations
The initial implementation reads the entire file into memory. File-size limits, more robust PDF error handling, and persistent storage remain future improvements.


## 3. Libraries, Functions, and Arguments: PDF Upload

### A. Libraries and Modules Used

| Library / Module | Import | Purpose |
|---|---|---|
| FastAPI | `FastAPI` | Creates the web application and registers API routes |
| FastAPI | `UploadFile` | Receives uploaded files |
| FastAPI | `File` | Declares a request parameter as a file upload |
| FastAPI | `HTTPException` | Raises an HTTP error response |
| pypdf | `PdfReader` | Reads PDF documents and extracts text |
| Python standard library | `io.BytesIO` | Treats bytes in memory like a file object |

**Why use these libraries?**

FastAPI handles HTTP requests, `pypdf` handles PDF reading, and Python's `io` module lets us process the uploaded PDF in memory without first saving it to disk.

### B. Functions and Decorators

#### 1. `@app.post("/upload-pdf")`

**Library:** FastAPI

**Type:** Decorator

Registers the function below it as the handler for HTTP POST requests to `/upload-pdf`.

**Argument:**
- `"/upload-pdf"`: The URL path assigned to the endpoint.

**Why POST?** Uploading a file sends data to the server, so POST is appropriate.

#### 2. `upload_pdf(file: UploadFile = File(...))`

**Type:** Async endpoint function

- `file`: The parameter that receives the uploaded file.
- `UploadFile`: The type FastAPI uses for uploaded files.
- `File(...)`: Declares the file as a required request parameter.
- `...`: Python's `Ellipsis` object, used here by FastAPI to indicate that the parameter is required.

The function is declared with `async` because it performs asynchronous file reading.

#### 3. `HTTPException(status_code=400, detail="...")`

**Library:** FastAPI

Raises an HTTP error instead of continuing to process an unacceptable upload.

**Arguments:**
- `status_code=400`: Returns HTTP 400, meaning the request is invalid.
- `detail="..."`: Supplies an explanation of the error.

These are **keyword arguments**, because each value is assigned by parameter name. They are not positional arguments.

#### 4. `await file.read()`

**Object:** `UploadFile`

Reads the uploaded file's contents.

- `read()` takes no explicit arguments here.
- `await` waits for the asynchronous read operation to finish.
- The returned value is the file content as bytes.

The result is stored in `contents`.

#### 5. `contents.startswith(b"%PDF-")`

**Method:** Python's built-in `bytes.startswith()`

Checks whether the file's bytes begin with the common PDF signature `%PDF-`.

**Argument:**
- `b"%PDF-"`: A bytes literal representing the expected signature.

The `b` prefix means this is bytes, not a regular Python string.

This is only a basic signature check. It does not prove that the entire document is valid or safe.

#### 6. `BytesIO(contents)`

**Library:** Python's `io` module

Creates an in-memory binary stream from the uploaded bytes.

**Argument:**
- `contents`: The bytes returned by `await file.read()`.

`PdfReader` can read from this file-like object without requiring a physical PDF file.

#### 7. `PdfReader(BytesIO(contents))`

**Library:** `pypdf`

Creates a PDF reader for the uploaded document.

**Argument:**
- `BytesIO(contents)`: The in-memory stream containing the PDF bytes.

The reader exposes information about the document, including its pages.

#### 8. `pdf.pages`

**Object:** `Pdf

## 4. Testing the PDF Upload Endpoint

### Test environment
- Server: Uvicorn
- Interface: FastAPI interactive documentation at `/docs`
- Endpoint: `POST /upload-pdf`

### Test procedure
1. Started the backend using Uvicorn.
2. Opened the interactive API documentation.
3. Selected a PDF file using the upload control.
4. Submitted the request.
5. Inspected the HTTP status code and JSON response.

### Expected response
- `filename`: Uploaded filename.
- `pages`: Number of pages detected.
- `text`: Text extracted from the PDF.

### Actual result
- HTTP status code: [fill in]
- Upload successful: [yes/no]
- Page count correct: [yes/no]
- Extracted text present: [yes/no/not applicable]
- Errors or observations: [fill in]

### Conclusion
[Write one sentence based on the actual test result. Do not mark the endpoint as working until the test confirms it.]

## 5. Starting and Verifying the Backend

### Command used

```powershell
.\.venv\Scripts\python.exe -m uvicorn backend.Main:app --reload
```

### Observed results

- `pypdf` installed successfully.
- Installed version: `6.21.3`.
- Uvicorn started successfully at `http://127.0.0.1:8000`.
- FastAPI completed application startup.
- Requests to `/` returned `200 OK`.
- Requests to `/docs` and `/openapi.json` returned `200 OK`.
- Requests to `/favicon.ico` returned `404 Not Found`.

### Understanding the log

- `200 OK`: The HTTP request succeeded.
- `404 Not Found`: The requested path was not found. The favicon response is not evidence of a backend failure.
- `Application startup complete`: FastAPI finished starting.
- `--reload`: Uvicorn watches project files and restarts the application when changes are detected.

### Current status

The server is running and the existing application responds successfully.

The PDF upload endpoint still requires explicit verification through `POST /upload-pdf`. A successful server startup alone does not prove that every endpoint works.

## 6. Successful PDF Upload and Text Extraction

### Test objective
Verify that Bookify can receive a PDF through its API and extract readable text from it.

### Test environment
- Framework: FastAPI
- Server: Uvicorn
- Testing interface: FastAPI Swagger UI (`/docs`)
- Endpoint: `POST /upload-pdf`
- PDF library: `pypdf 6.21.3`

### Test procedure
1. Started the FastAPI server.
2. Opened `/docs` in the browser.
3. Selected a PDF using the upload control.
4. Submitted the request.
5. Inspected the response and extracted text.

### Actual result

| Check | Result |
|---|---|
| HTTP status | `200 OK` |
| Filename returned | Yes |
| Page count returned | Yes, 1 page |
| Text extracted | Yes |
| JSON response received | Yes |

### What this proves

The endpoint successfully received the uploaded PDF, read its contents, processed the document using `PdfReader`, extracted readable text, and returned the results as JSON.

The test confirms that the basic upload and extraction pipeline works for this particular PDF.

### What it does not prove

- All PDFs will be readable.
- Scanned documents will work without OCR.
- Large files can be processed safely.
- Uploaded files are saved permanently.
- Extracted text or book metadata is stored in SQLite.

### Engineering conclusion

**The first PDF upload and text-extraction test passed.** Bookify can now process a suitable uploaded PDF and return its extracted text. Persistent file storage, metadata storage, and more robust validation remain future work.

### Personal Remark
Something did happen, maja aaya, whenever refressing the site, two new lines appear on the terminal
INFO:     127.0.0.1:63460 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:62847 - "GET / HTTP/1.1" 200 OK
well, the moksha file even had a qr, it couldnt read that, but then at the end of the day, atleast it reads text from the pdf, very happy and maje maje mei added a heavy ass pdf
also, the UI which was visible, very new to me, but it was Fast API'sSWageer UI, idk what, but maja ayaa


## 6. Successful PDF Upload and Text Extraction

### Test objective
Verify that Bookify can receive a PDF through its API and extract readable text from it.

### Test environment
- Framework: FastAPI
- Server: Uvicorn
- Testing interface: FastAPI Swagger UI (`/docs`)
- Endpoint: `POST /upload-pdf`
- PDF library: `pypdf 6.21.3`

### Test procedure
1. Started the FastAPI server.
2. Opened `/docs` in the browser.
3. Selected a PDF using the upload control.
4. Submitted the request.
5. Inspected the response and extracted text.

### Actual result

| Check | Result |
|---|---|
| HTTP status | `200 OK` |
| Filename returned | Yes |
| Page count returned | Yes, 1 page |
| Text extracted | Yes |
| JSON response received | Yes |

### What this proves

The endpoint successfully received the uploaded PDF, read its contents, processed the document using `PdfReader`, extracted readable text, and returned the results as JSON.

The test confirms that the basic upload and extraction pipeline works for this particular PDF.

### What it does not prove

- All PDFs will be readable.
- Scanned documents will work without OCR.
- Large files can be processed safely.
- Uploaded files are saved permanently.
- Extracted text or book metadata is stored in SQLite.

### Engineering conclusion

**The first PDF upload and text-extraction test passed.** Bookify can now process a suitable uploaded PDF and return its extracted text. Persistent file storage, metadata storage, and more robust validation remain future work.

### Personal Remark
It did work on the heavy book, added the intro to ml using python, some 6000kb book, 392 pages and it did extract the text, idk gotta see what all it extracted but yes, the page count 392, is correct, NICE
